# MobileNet-UNet latent vs physical — no reconstruction (UIEB)

Kaggle runner for the two urgent no-reconstruction variants: `learnable_latent_mobilenet_unet` and `parameterized_physics_mobilenet_unet`. It follows the existing fixed UIEB 800/90 protocol: 720 train, 80 validation, and the final 90 pairs held out for testing. Checkpoints are written after every epoch and incomplete runs resume automatically.

Before running: enable a GPU and Internet, attach UIEB `raw-890` and `reference-890`, and make sure `REPO_BRANCH` contains the two registered MobileNet models.

In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/heniath/underwater-image-enhancement.git'
REPO_BRANCH = 'learnable-physics-extractor'
REPO = Path('/kaggle/working/underwater-image-enhancement')

if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch', '--depth', '1',
        REPO_URL, str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)

if importlib.util.find_spec('kornia') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.7.3'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
src = str(REPO / 'src')
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
models = subprocess.check_output(['uwir-profile', '--list'], text=True)
for required in ('learnable_latent_mobilenet_unet', 'parameterized_physics_mobilenet_unet'):
    assert required in models, f'{required} is missing from {REPO_BRANCH}; push the MobileNet changes first'
print('Repository commit:', commit)

In [ ]:
import torch

raw_candidates = [
    Path('/kaggle/input/datasets/larjeck/uieb-dataset-raw/raw-890'),
    *Path('/kaggle/input').glob('**/raw-890'),
]
reference_candidates = [
    Path('/kaggle/input/datasets/larjeck/uieb-dataset-reference/reference-890'),
    *Path('/kaggle/input').glob('**/reference-890'),
]
RAW_DIR = next((path for path in raw_candidates if path.is_dir()), None)
REFERENCE_DIR = next((path for path in reference_candidates if path.is_dir()), None)
assert RAW_DIR is not None, 'UIEB raw-890 folder was not found'
assert REFERENCE_DIR is not None, 'UIEB reference-890 folder was not found'

UIEB_ROOT = Path('/kaggle/working/UIEB')
UIEB_ROOT.mkdir(parents=True, exist_ok=True)
for name, target in [('raw-890', RAW_DIR), ('reference-890', REFERENCE_DIR)]:
    link = UIEB_ROOT / name
    if link.is_symlink() and link.resolve() != target.resolve():
        link.unlink()
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS > 0, 'Enable a Kaggle GPU accelerator'
SMOKE = False  # True: one epoch and one seed
EPOCHS = 1 if SMOKE else 100
seed_spec = os.environ.get('UWIR_SEEDS_TO_RUN')
SEEDS = [int(value.strip()) for value in seed_spec.split(',')] if seed_spec else [0]
if SMOKE:
    SEEDS = SEEDS[:1]
BATCH_SIZE = 4
CROP_SIZE = 256
WORKERS = 2
ABLATIONS = {
    'mobile_latent_no_reconstruction': dict(
        model='learnable_latent_mobilenet_unet', reconstruction=0.0, smoothness=0.0),
    'mobile_physical_no_reconstruction': dict(
        model='parameterized_physics_mobilenet_unet', reconstruction=0.0, smoothness=0.01),
}
OUTPUT_ROOT = Path('/kaggle/working/mobilenet_physics_no_recon_uieb')
PREVIOUS_OUTPUT_ROOT = globals().get('PREVIOUS_OUTPUT_ROOT') or os.environ.get('UWIR_PREVIOUS_OUTPUT_ROOT')
if PREVIOUS_OUTPUT_ROOT:
    previous = Path(PREVIOUS_OUTPUT_ROOT)
    assert previous.is_dir(), f'Previous output not found: {previous}'
    if previous.resolve() != OUTPUT_ROOT.resolve():
        shutil.copytree(previous, OUTPUT_ROOT, dirs_exist_ok=True)
        print('Copied previous output:', previous, '->', OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda')
print('Dataset:', UIEB_ROOT, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])
print('Mode:', 'smoke' if SMOKE else 'full', '| epochs:', EPOCHS, '| seeds:', SEEDS)
print('Ablations:', list(ABLATIONS))

In [ ]:
import json
import random
import time

import numpy as np
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import Compose, ToTensor

from uwir.cli.train import EarlyStopping, load_ckpt, save_ckpt, train_epoch, val_loss_epoch
from uwir.data.datasets import UIEBDataset
from uwir.losses import CompositeLoss, PhysicsConsistentLoss
from uwir.metrics import evaluate_loader
from uwir.models import build_model
from uwir.training.schedulers import CosineAnnealingRestartLR

transform = Compose([ToTensor()])
train_base = UIEBDataset(str(UIEB_ROOT), transform=transform, augment=True, img_size=CROP_SIZE)
eval_base = UIEBDataset(str(UIEB_ROOT), transform=transform, augment=False, img_size=CROP_SIZE)
assert len(train_base) == 890, f'Expected 890 UIEB pairs, found {len(train_base)}'
TEST_INDICES = list(range(800, 890))
test_dataset = Subset(eval_base, TEST_INDICES)

def collate_rgb(batch):
    return torch.stack([item[0] for item in batch]), torch.stack([item[1] for item in batch])

def make_loader(dataset, *, shuffle, batch_size=BATCH_SIZE):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=WORKERS,
        pin_memory=True, drop_last=shuffle, collate_fn=collate_rgb, persistent_workers=False,
    )

print('UIEB split: train/validation pool=800, held-out test=90')

In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

def atomic_json_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2))
    os.replace(temporary, path)

def run_is_complete(run_dir):
    marker = run_dir / 'run_complete.json'
    return marker.is_file()

def train_run(config_name, config, seed):
    run_dir = OUTPUT_ROOT / config_name / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path = run_dir / 'best_model.pth'
    last_path = run_dir / 'last_model.pth'
    if best_path.is_file() and run_is_complete(run_dir):
        print(f'{config_name} seed {seed}: already complete')
        return best_path

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    order = torch.randperm(800, generator=torch.Generator().manual_seed(seed)).tolist()
    val_indices = sorted(order[:80])
    train_indices = sorted(order[80:])
    train_loader = make_loader(Subset(train_base, train_indices), shuffle=True)
    val_loader = make_loader(Subset(eval_base, val_indices), shuffle=False)

    model = build_model(config['model']).to(DEVICE)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    if NUM_GPUS > 1:
        model = torch.nn.DataParallel(model)
    enhancement_loss = CompositeLoss(lambda_l1=1.0, lambda_perc=1.0, lambda_ssim=0.0, device=DEVICE)
    criterion = PhysicsConsistentLoss(
        enhancement_loss, lambda_reconstruction=config['reconstruction'],
        lambda_depth_smoothness=config['smoothness'],
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
    scheduler = CosineAnnealingRestartLR(
        optimizer, periods=[EPOCHS], restart_weights=[1.0], eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=True, init_scale=1024.0)
    early_stopping = EarlyStopping(patience=20, min_delta=1e-4, mode='max')
    history = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': [],
               'reconstruction': [], 'depth_smoothness': [], 'lr': []}
    best_psnr, best_ssim, start_epoch = float('-inf'), float('-inf'), 1
    atomic_json_save(
        {'seed': seed, 'train': train_indices, 'validation': val_indices, 'test': TEST_INDICES},
        run_dir / 'split_manifest.json')

    if last_path.is_file():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        bare_model = model.module if isinstance(model, torch.nn.DataParallel) else model
        bare_model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler'])
        scaler.load_state_dict(checkpoint['scaler'])
        history = checkpoint['history']
        best_psnr, best_ssim = checkpoint['best_psnr'], checkpoint['best_ssim']
        state = checkpoint.get('early_stopping', {})
        early_stopping.best = state.get('best', best_psnr)
        early_stopping.counter = state.get('counter', 0)
        start_epoch = int(checkpoint['epoch']) + 1
        print(f'{config_name} seed {seed}: resume at epoch {start_epoch}')

    print(f'\n{config_name} | seed={seed} | params={parameter_count / 1e6:.3f}M')
    if start_epoch > EPOCHS:
        atomic_json_save({'epoch': start_epoch - 1, 'reason': 'epochs_complete'},
                         run_dir / 'run_complete.json')
        return best_path
    stopped = False
    for epoch in range(start_epoch, EPOCHS + 1):
        started = time.time()
        train_loss, parts = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler=scaler)
        val_loss = val_loss_epoch(model, val_loader, criterion, DEVICE, amp_enabled=True)
        metrics, _ = evaluate_loader(model, val_loader, DEVICE)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        for key, value in [('train_loss', train_loss), ('val_loss', val_loss),
                           ('val_psnr', metrics['psnr']), ('val_ssim', metrics['ssim']),
                           ('reconstruction', parts.get('reconstruction', 0.0)),
                           ('depth_smoothness', parts.get('depth_smoothness', 0.0)), ('lr', lr)]:
            history[key].append(value)
        improved = (metrics['psnr'], metrics['ssim']) > (best_psnr, best_ssim)
        if improved:
            best_psnr, best_ssim = metrics['psnr'], metrics['ssim']
            save_ckpt(model, optimizer, epoch, {**metrics, 'val_loss': val_loss}, str(best_path))
        stopped = early_stopping(metrics['psnr'])
        bare_model = model.module if isinstance(model, torch.nn.DataParallel) else model
        payload = {
            'epoch': epoch, 'model': bare_model.state_dict(), 'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'history': history,
            'best_psnr': best_psnr, 'best_ssim': best_ssim,
            'early_stopping': {'best': early_stopping.best, 'counter': early_stopping.counter},
        }
        atomic_torch_save(payload, last_path)
        atomic_json_save(history, run_dir / 'training_history.json')
        if epoch == 1 or epoch % 5 == 0 or improved:
            print(f'{epoch:03d}/{EPOCHS} train={train_loss:.4f} val={val_loss:.4f} '
                  f'PSNR={metrics["psnr"]:.3f} SSIM={metrics["ssim"]:.4f} '
                  f'lr={lr:.2e} {time.time() - started:.1f}s' + (' BEST' if improved else ''))
        if stopped:
            print('Early stopping at epoch', epoch)
            break
    atomic_json_save({'epoch': epoch, 'reason': 'early_stop' if stopped else 'epochs_complete'},
                     run_dir / 'run_complete.json')
    return best_path

In [ ]:
best_checkpoints = {}
for config_name, config in ABLATIONS.items():
    best_checkpoints[config_name] = {}
    for seed in SEEDS:
        best_checkpoints[config_name][seed] = train_run(config_name, config, seed)
        gc.collect()
        torch.cuda.empty_cache()
best_checkpoints

In [ ]:
test_loader = make_loader(test_dataset, shuffle=False)
test_results = {}
for config_name, config in ABLATIONS.items():
    test_results[config_name] = {}
    for seed, checkpoint in best_checkpoints[config_name].items():
        model = build_model(config['model']).to(DEVICE)
        epoch, validation = load_ckpt(str(checkpoint), model, device=str(DEVICE))
        metrics, count = evaluate_loader(model, test_loader, DEVICE)
        test_results[config_name][str(seed)] = {
            'model': config['model'], 'reconstruction_weight': config['reconstruction'],
            'smoothness_weight': config['smoothness'], 'epoch': epoch, 'count': count,
            'validation': validation, 'test': metrics,
        }
        print(f'{config_name} | seed {seed} | epoch {epoch} | n={count} | '
              f'PSNR={metrics["psnr"]:.3f} SSIM={metrics["ssim"]:.4f} '
              f'CIEDE2000={metrics["ciede2000"]:.3f} UCIQE={metrics["uciqe"]:.3f} '
              f'UIQM={metrics["uiqm"]:.3f}')
atomic_json_save(test_results, OUTPUT_ROOT / 'test_results.json')

metric_names = ['psnr', 'ssim', 'ciede2000', 'uciqe', 'uiqm']
summary = {}
for config_name, runs in test_results.items():
    summary[config_name] = {}
    for metric in metric_names:
        values = np.array([run['test'][metric] for run in runs.values()])
        summary[config_name][metric] = {
            'mean': float(values.mean()),
            'std': float(values.std(ddof=1)) if len(values) > 1 else 0.0,
        }
atomic_json_save(summary, OUTPUT_ROOT / 'ablation_summary.json')
archive = shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT)
print(json.dumps(summary, indent=2))
print('Outputs:', OUTPUT_ROOT)
print('Downloadable archive:', archive)